In [1]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/var/folders/gg/tq3320f11lggvg4t9w5_kdwc0000gn/T/ipykernel_47192/2300357140.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

# 1. 데이터 로드 및 벡터 저장소 연결
try:
    # 가벼운 텍스트 파서(TextLoader)를 강제 지정하여 문서를 읽어옵니다.
    loader = DirectoryLoader('./data', glob="**/*.txt", loader_cls=TextLoader)
    docs = loader.load()
    
    # Nomic 모델로 텍스트를 벡터로 변환하여 Chroma DB에 저장합니다.
    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings, persist_directory="./chroma_db")
    retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
    print("✅ 2. Vector DB 로드 성공")
    
except Exception as e:
    print(f"⚠️ DB 생성 중 에러 발생: {e}")

✅ 2. Vector DB 로드 성공


In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. 로컬 LLM(llama3.1) 연결
llm = Ollama(model="llama3.1:latest")

# 2. 프롬프트 템플릿 세팅
prompt = ChatPromptTemplate.from_template("""
당신은 스마트홈 인프라 매니저입니다. 제공된 <문서>에만 기반하여 질문에 답하세요.

<문서>
{context}
</문서>

질문: {input}
""")

# 3. 검색된 문서를 텍스트로 이어붙여주는 헬퍼 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. 최신 LCEL 문법으로 파이프라인(Chain) 조립
rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
print("✅ 3. RAG 파이프라인 조립 완료")

# 5. 실제 스마트홈 제어 테스트 질문 투척!
print("\n🤖 AI 추론 중...")
response = rag_chain.invoke("세탁실 들어갈 때 조도 기준을 잡아야 하는데 우리집 어떤 센서의 ID를 가져와서 조건문을 짜야 돼?")
print(f"\n💡 답변: {response}")

/var/folders/gg/tq3320f11lggvg4t9w5_kdwc0000gn/T/ipykernel_47754/1212933604.py:7: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1:latest")


✅ 3. RAG 파이프라인 조립 완료

🤖 AI 추론 중...

💡 답변: 드레스룸 조도 센서입니다. 엔티티 ID는 sensor.dressroom_sensor_mmwave_illuminance 입니다.

조건문은 다음과 같이 작성할 수 있습니다:

```
if sensor.dressroom_sensor_mmwave_illuminance.state <= 25:
    light.sewage_room_switch_on.on()
```

세탁실 들어갈 때 조도 센서가 없으므로, 인접한 드레스룸의 조도 센서 값을 추정하여 조명 켜기 조건을 설정했습니다.
